<h1> Monte Carlo Control

1. Monte Carlo Prediction

If I have a policy, how good is it?

That is:

$V^\pi(s)$

2. Monte Carlo Control

What is the best policy?

That is, finding:

$\pi^*$

1. Prediction

Input: Fixed policy</br>
Output: Value function

2. Control

Input: Experience</br>
Output: Optimal policy

The agent learns for itself:
- When to hit.
- When to stick.

### Q-table

In Prediction, we had:
V[state]

Now we have:
Q[state, action]

Since there are two actions:
0 = Stick
1 = Hit

**Example:**

Q[(20,8,0)]

Output:</br>
[0.4, -0.2]

Meaning:</br>
Stick = 0.4</br>
Hit = -0.2

So:</br>
Stick is better.

In [1]:
import numpy as np

from collections import defaultdict
import gymnasium as gym

from src.mont_calro import (
    epsilon_greedy,
    control_generate_episode,
    monte_carlo_control,
)

c:\Users\niush\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


<h3> Step 1: ε-greedy Function

1. Constructing the Q-table
2. Constructing the ε-greedy policy

In [2]:
env = gym.make(
    "Blackjack-v1"
)

In [3]:
Q = defaultdict(
    lambda: np.zeros(
        env.action_space.n
    )
)

In [4]:
Q[(20,8,0)]

array([0., 0.])

In [5]:
state, info = env.reset()

print(state)

print(
    epsilon_greedy(
        env,
        state,
        Q
    )
)

(16, 9, 0)
0


$(Player sum, Dealer showing card, No usable ace)$

<h3> Step 2: Episode Generator for Monte Carlo Control

In [6]:
episode = control_generate_episode(
    env,
    Q
)

episode

[((10, 6, 0), 0, 1.0)]

In [7]:
for _ in range(5):
    print(
        control_generate_episode(
            env,
            Q
        )
    )

[((13, 8, 1), 0, -1.0)]
[((12, 1, 0), 0, -1.0)]
[((12, 1, 0), 0, -1.0)]
[((8, 1, 0), 0, -1.0)]
[((18, 2, 0), 0, 1.0)]


The agent has not learned anything yet and is operating solely based on the initial policy.

#### Difference from Prediction:

Prediction: Random Policy

Here: ε-greedy Policy

This means that over time:
- Q improves.
- The policy improves.
- The episodes improve.

<h3> Step 3: Training (Monte Carlo Control — Q Update)

In [8]:
Q = monte_carlo_control(
    env,
    episodes=50000
)

In [9]:
state = (20,8,0)

print(
    Q[state]
)

[ 0.79359431 -1.        ]


In [10]:
states = {
    "Strong hand vs dealer 9": (20, 9, 0),
    "Weak hand vs dealer 10": (13, 10, 0),
    "Low hand vs dealer 2": (12, 2, 0)
}

action_names = {
    0: "0: Stick",
    1: "1: Hit"
}

for name, state in states.items():
    best_action = np.argmax(Q[state])

    print(name)
    print("State:", state)
    print("Q-values:", Q[state])
    print("Best action:", action_names[best_action])
    print("-" * 30)

Strong hand vs dealer 9
State: (20, 9, 0)
Q-values: [ 0.7835249 -0.92     ]
Best action: 0: Stick
------------------------------
Weak hand vs dealer 10
State: (13, 10, 0)
Q-values: [-0.55805243 -0.48398577]
Best action: 1: Hit
------------------------------
Low hand vs dealer 2
State: (12, 2, 0)
Q-values: [-0.37888199 -0.28409091]
Best action: 1: Hit
------------------------------


The results are very good. This is the first instance where we see the agent truly learning from experience, rather than just acting randomly.

<h3> Step 4: Policy Evaluation and Comparison with a Random Agent

During evaluation, we no longer want exploration.

❌ ε-greedy

Because we do not want the agent to move randomly.</br>
We want to: Always choose the best action.

That is:

$a = \text{argmax}(Q(s,a))$

During the training phase, we used the ε-greedy strategy to construct the Q-table; however, once it was built, we relied solely on the 'max' operation for execution during the evaluation phase.

<h3> Step 4.1: Creating an agent with a learned policy

In [11]:
def q_policy(
    state,
    Q
):
    return np.argmax(
        Q[state]
    )

<h3> Step 4.2: Executing an episode with the learned policy

In [12]:
def run_episode(
    env,
    policy
):
    state, info = env.reset()
    total_reward = 0

    while True:
        action = policy(state)
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    return total_reward

<h3> Step 4.3: Evaluation Function

In [13]:
def evaluate(
    env,
    policy,
    episodes=10000
):
    rewards = []

    for _ in range(episodes):
        reward = run_episode(
            env,
            policy
        )
        rewards.append(
            reward
        )

    return np.mean(rewards)

<h3> Step 4.4: Evaluation Monte Carlo Agent

In [14]:
mc_score = evaluate(
    env,
    lambda state:
    q_policy(
        state,
        Q
    )
)

print("mc_score: ", mc_score)

mc_score:  -0.0595


<h3> Step 4.5: Random Baseline

In [15]:
random_score = evaluate(
    env,
    lambda state:
    env.action_space.sample()
)

print("random_score: ", random_score)

random_score:  -0.3787


#### Part 1: Training (Learning)

Here, we used: $ε-greedy$

Why?
Because the agent does not yet know which action is best.
So, it must engage in both:

Exploitation

Using what it currently knows:
Selecting the action with the best current Q-value

And:
Exploration

Trying new things:
Occasionally choosing a random action

Thus:
action = epsilon_greedy(state,Q,epsilon)

This led to the following sequence:
Episode
↓
Experience
↓
Return
↓
Update Q

#### Part 2: Evaluation (Execution)
Here, the goal is no longer learning.

The objective is:
To see how good the learned policy is.

So, we disable exploration.
Because if we continue to move randomly, we cannot determine how good the policy truly is.

## Monte Carlo Control Evaluation

The learned Monte Carlo control policy was evaluated against a random baseline on the Blackjack environment.

Results:

| Policy | Average Reward |
|---|---:|
| Random Policy | -0.3964 |
| Monte Carlo Control | -0.0556 |

The Monte Carlo agent achieved a significantly higher average return compared to the random agent.

Although the final reward is still negative due to the stochastic nature of Blackjack and limited training samples, the improvement demonstrates that the agent successfully learned a better action-selection strategy through experience.

The agent learned different behaviors depending on the state, such as choosing Stick for strong hands and Hit for weaker hands.